In [1]:
import pandas as pd
import numpy as np
import os

# 1. Setup the path (Same as before)
data_dir = '../dataset/archive/2nd_test/2nd_test'
filenames = os.listdir(data_dir)
filenames.sort()

# 2. Create an empty list to hold our data rows
data_rows = []

print(f"Starting feature extraction on {len(filenames)} files. Please wait...")

# 3. The Loop: Go through EVERY file
for index, file in enumerate(filenames):
    
    # Optional: Print progress every 100 files so you know it's working
    if index % 100 == 0:
        print(f"Processing file {index}...")

    # Read the raw file
    path = os.path.join(data_dir, file)
    df_raw = pd.read_csv(path, sep='\t', header=None)
    
    # 4. Extract Features for ALL 4 Bearings
    # We create a dictionary (key-value pair) for this single file
    row_features = {'filename': file}
    
    for i in range(4): # Loop through Bearing 1 to 4
        col_name = i  # 0, 1, 2, 3
        bearing_num = i + 1
        
        data = df_raw[col_name]
        
        # Calculate the Physics Metrics
        row_features[f'B{bearing_num}_Mean'] = np.mean(data)
        row_features[f'B{bearing_num}_Std'] = np.std(data)
        row_features[f'B{bearing_num}_Skew'] = data.skew()
        row_features[f'B{bearing_num}_Kurtosis'] = data.kurtosis()

    # Add this completed row to our list
    data_rows.append(row_features)

# 5. Convert the list into a Clean Table (DataFrame)
df_final = pd.DataFrame(data_rows)

print("--- Extraction Complete ---")
print(f"New Dataset Shape: {df_final.shape}")
print("Here are the first 5 rows of your engineered data:")
display(df_final.head())

Starting feature extraction on 984 files. Please wait...
Processing file 0...
Processing file 100...
Processing file 200...
Processing file 300...
Processing file 400...
Processing file 500...
Processing file 600...
Processing file 700...
Processing file 800...
Processing file 900...
--- Extraction Complete ---
New Dataset Shape: (984, 17)
Here are the first 5 rows of your engineered data:


,filename,B1_Mean,B1_Std,B1_Skew,B1_Kurtosis,B2_Mean,B2_Std,B2_Skew,B2_Kurtosis,B3_Mean,B3_Std,B3_Skew,B3_Kurtosis,B4_Mean,B4_Std,B4_Skew,B4_Kurtosis
0,2004.02.12.10.32.39,-0.010196,0.073475,0.084000,0.629209,-0.012695,0.090053,0.126924,0.507217,-0.014541,0.108434,0.204855,3.214152,-0.010026,0.053166,-0.022082,0.066268
1,2004.02.12.10.42.39,-0.002585,0.075338,0.052146,0.648742,-0.002561,0.093384,0.070094,0.253369,-0.002461,0.109790,-0.023855,1.395884,-0.003784,0.055973,0.001583,0.107859
2,2004.02.12.10.52.39,-0.002484,0.076189,0.032810,0.513894,-0.001695,0.093703,0.096590,0.311158,-0.001595,0.109849,0.056565,2.640886,-0.003485,0.056037,0.070454,0.257592
3,2004.02.12.11.02.39,-0.002277,0.078691,0.041489,1.158529,-0.002393,0.092916,0.105842,0.235691,-0.003148,0.110622,0.033553,2.683727,-0.003741,0.056684,-0.036330,0.806680
4,2004.02.12.11.12.39,-0.002404,0.078437,0.028226,0.603617,-0.001559,0.095335,0.097973,0.226657,-0.001158,0.107499,-0.002892,1.579073,-0.002703,0.056777,0.019009,0.139281


In [2]:
# 1. Let's define the "Answer Key"
# We'll rely on the timeline.
# First 400 files = Healthy (0)
# Last 100 files = Faulty (1)
# The middle files? We'll ignore them for training to avoid confusing the AI.

# Create the labels
healthy_df = df_final.iloc[0:400].copy()
healthy_df['Status'] = 0  # 0 means Healthy

faulty_df = df_final.iloc[-100:].copy()
faulty_df['Status'] = 1   # 1 means Faulty

# Combine them back together
training_data = pd.concat([healthy_df, faulty_df])

print("--- Labeling Complete ---")
print(f"Healthy Samples: {len(healthy_df)}")
print(f"Faulty Samples: {len(faulty_df)}")
print(f"Total Training Data: {len(training_data)}")

--- Labeling Complete ---
Healthy Samples: 400
Faulty Samples: 100
Total Training Data: 500


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Separate the "Questions" (X) from the "Answers" (y)
# We drop 'filename' because the name doesn't cause failure.
# We drop 'Status' from X because that's the answer!
X = training_data.drop(['filename', 'Status'], axis=1)
y = training_data['Status']

# 2. Split: 80% for Studying, 20% for the Exam
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Build the Brain (Random Forest)
model = RandomForestClassifier(n_estimators=100) # 100 "Decision Trees"

# 4. Train it
print("Training the model...")
model.fit(X_train, y_train)

# 5. Test it
predictions = model.predict(X_test)

# 6. Report Card
accuracy = accuracy_score(y_test, predictions)
print(f"\n--- RESULTS ---")
print(f"Model Accuracy: {accuracy * 100:.2f}%")
print("\nDetailed Report:")
print(classification_report(y_test, predictions))

Training the model...

--- RESULTS ---
Model Accuracy: 100.00%

Detailed Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        76
           1       1.00      1.00      1.00        24

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [4]:
import joblib

# 1. Save the trained model to a file
# We'll save it in that empty 'models' folder we made at the start
model_filename = '../models/bearing_classifier.pkl'
joblib.dump(model, model_filename)

print(f"Success! Model saved to: {model_filename}")
print("You can now download this file and send it to anyone.")

Success! Model saved to: ../models/bearing_classifier.pkl
You can now download this file and send it to anyone.
